In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from glm.extras.participation_intensity import (
    CoverageSimulation,
    ParameterSampleRanges,
    ParticipationIntensityFunction,
    ParticipationIntensityModel,
    sample_data,
)

from scipy.special import lambertw as W
from scipy.special import wrightomega as ω

In [ ]:
IMG_DIR = "~/tmp"

In [ ]:
np.random.seed(154)

### Display plots of special functions

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))

ts = np.linspace(-1/np.e + 0.001, 4.0, num=1000)
ax.plot(ts, W(ts), linewidth=3, label=r"$W$")

ts = np.linspace(-4.0, 4.0, num=1000)
ax.plot(ts, ω(ts), linewidth=3, label=r"$\omega$")

ax.axhline(0.0, linestyle='--', color="grey", alpha=0.5)

ax.legend()

fig.savefig(IMG_DIR + "pif_special_functions.png",  bbox_inches="tight")

# Participation–Intensity Coditional Mean

#### Nice Example

In [ ]:
f = ParticipationIntensityFunction(-1.0, 7.5, -0.5, -2.0)

fig, ax = plt.subplots(figsize=(14, 3))

xs = np.linspace(0.0, 5.0, num=250)
ax.plot(xs, f(xs))
ax.axvline(f.argmax(), linestyle='--', color='grey')
ax.scatter(f.argmax(), f.max(), s=125)

#ax.set_xticks([])
ax.set_yticks([])

ax.set_title(str(f))

fig.savefig(IMG_DIR + "pif_participation_intensity_function.png",  bbox_inches="tight")

#### Approaching β = 0

In [ ]:
fs = [
    ParticipationIntensityFunction(-1.0, 7.5, -0.5, β) 
    for β in -np.logspace(np.log10(2.0), np.log10(0.1), num=10)
]
xmax = max(f.argmax() for f in fs)


cmap = plt.get_cmap("viridis")
colors = cmap(np.linspace(0, 1, len(fs)))

fig, ax = plt.subplots(figsize=(14, 3))
xs = np.linspace(0.0, xmax + 1.0, num=250)

for f, c in zip(fs, colors):
    ax.plot(xs, f(xs), label=f"β={f.β:2.2f}", color=c)
    ax.axvline(f.argmax(), linestyle='--', color=c)
    ax.scatter(f.argmax(), f.max(), s=125, color=c)
ax.legend()
ax.set_title("Participation-Intensity Functions as  β ↗ 0")

fig.savefig(IMG_DIR + "pif_participation_intensity_vary_beta.png",  bbox_inches="tight")

#### Approaching b = 0

In [ ]:
fs = [
    ParticipationIntensityFunction(-1.0, b, -0.5, -2.0) 
    for b in np.logspace(np.log10(10.0), np.log10(0.2), num=10)
]
xmax = max(f.argmax() for f in fs)

cmap = plt.get_cmap("viridis")
colors = cmap(np.linspace(0, 1, len(fs)))

fig, ax = plt.subplots(figsize=(14, 3))
xs = np.linspace(0.0, xmax + 1.0, num=250)

for f, c in zip(fs, colors):
    ax.plot(xs, f(xs), label=f"b={f.b:2.2f}", color=c)
    ax.axvline(f.argmax(), linestyle='--', color=c)
    ax.scatter(f.argmax(), f.max(), s=125, color=c)
ax.legend()
ax.set_title("Participation-Intensity Functions as  b ↘ 0")

fig.savefig(IMG_DIR + "pif_participation_intensity_vary_b.png",  bbox_inches="tight")

## Sample Parameters

In [ ]:
def sample_fs(ranges: ParameterSampleRanges, n: int) -> list[ParticipationIntensityFunction]:
    fs: list[ParticipationIntensityMean] = []
    for i in range(n):
        a = np.random.uniform(low=ranges.a[0], high=ranges.a[1])
        b = np.random.uniform(low=ranges.b[0], high=ranges.b[1])
        α = np.random.uniform(low=ranges.α[0], high=ranges.α[1])
        β = np.random.uniform(low=ranges.β[0], high=ranges.β[1])
        fs.append(ParticipationIntensityFunction(a, b, α, β))
    return fs

In [ ]:
rgs = ParameterSampleRanges(
    a=(-0.1, 0.0),
    b=(0.1, 1.2),
    α=(-1.0, -0.25),
    β=(-2.0, -0.1)
)
fs = sample_fs(ranges=rgs, n=10)
fs = list(sorted(fs, key=lambda m: -m.max()))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))

xmax = max(m.argmax() for m in fs) + 1.0
xs = np.linspace(0.0, xmax, num=250)

cmap = plt.get_cmap("viridis")
colors = cmap(np.linspace(0, 1, len(fs)))

for m, color in zip(fs, colors):
    ax.plot(xs, m(xs), color=color)
    ax.scatter(m.argmax(), m.max(), s=125, color=color)
    ax.axvline(m.argmax(), linestyle='--', color='grey', alpha=0.25)
ax.set_title("Various Participation–Intensity Means with Maxima")

# Sample Dataset

In [ ]:
xs = np.random.uniform(0.0, 5.0, size=250)

f = ParticipationIntensityFunction(-1.0, 7.5, -0.5, -2.0)
X = sample_data(f, xs)
X.loc[X["d"] == 0, "y"] = 0.0
model = ParticipationIntensityModel().fit(X)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))

xs = np.linspace(0.0, 5.0, num=250)

ax.plot(xs, f(xs), label="True f")
ax.scatter(f.argmax(), f.max(), s=125)
ax.axvline(f.argmax(), linestyle='--')

ax.plot(xs, model.mean_(xs), linestyle=':', color='orange')
ax.scatter(model.mean_.argmax(), model.mean_.max(), s=75, color='orange')
ax.axvline(model.mean_.argmax(), linestyle='--', color='orange', label="Estimated f")
# ax.axvspan(
#     model.argmax_() - 1.96 * model.argmax_se_(),
#     model.argmax_() + 1.96 * model.argmax_se_(),
#     color='orange',
#     alpha=0.25,
# )

ax.legend()
ax.set_title("Participation–Intensity Function with Fit from Sampled Data (N=250)")

fig.savefig(IMG_DIR + "pif_participation_intensity_fit.png",  bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))

xs = np.linspace(0.0, 3.0, num=250)

models: list[ParticipationIntensityModel] = []
for _ in range(100):
    ts = np.random.uniform(0.0, 3.0, size=250)
    X = sample_data(f, ts)
    models.append(ParticipationIntensityModel().fit(X))

for model in models[:20]:
    ax.plot(xs, model.mean_(xs), linestyle=':', color='orange', alpha=0.33)
    ax.scatter(model.mean_.argmax(), model.mean_.max(), s=75, color='orange', alpha=0.75)
    ax.axvline(model.mean_.argmax(), linestyle='--', color='orange', alpha=0.33)

ax.plot(xs, f(xs), label="True f")
ax.scatter(f.argmax(), f.max(), s=125)
ax.axvline(f.argmax(), linestyle='--')

# Tick marks along the x axis at each plotted fit's argmax.
ax.set_xticks([model.mean_.argmax() for model in models[:20]], minor=True)
ax.tick_params(axis='x', which='minor', length=10, width=1.5, color='orange')

ax.set_title("Participation–Intensity Function with Repeated Fits from Sampled Data (N = 250)")

fig.savefig(IMG_DIR + "pif_participation_intensity_fits.png",  bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))

ax.axhline(f.argmax(), linestyle='--', label="True argmax")

for i, model in enumerate(models):
    lgn = "Estiamted argmax" if i == 0 else None
    ax.scatter(i, model.argmax_(), color='orange', label=lgn)

ax.legend()

#### With Intervals

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))

xs = np.linspace(0.0, 3.0, num=250)

ax.plot(xs, f(xs), label="True f")
ax.scatter(f.argmax(), f.max(), s=125)
ax.axvline(f.argmax(), linestyle='--')

ax.plot(xs, model.mean_(xs), linestyle=':', color='orange')
ax.scatter(model.mean_.argmax(), model.mean_.max(), s=75, color='orange')
ax.axvline(model.mean_.argmax(), linestyle='--', color='orange', label="Estimated f")
ax.axvspan(
    model.argmax_() - 1.96 * model.argmax_se_(),
    model.argmax_() + 1.96 * model.argmax_se_(),
    color='orange',
    alpha=0.25,
)

ax.legend()
ax.set_title("Participation–Intensity Function with Fit from Sampled Data")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))
COLORS = {0: 'darkgrey', 1: 'orange'}

true_argmax = f.argmax()
ax.axhline(true_argmax, linestyle='--', label="True argmax")

for i, model in enumerate(models):
    lgn = "Estiamted argmax" if i == 0 else None
    amax = model.argmax_()
    upper = model.argmax_() + 2*model.argmax_se_()
    lower = model.argmax_() - 2*model.argmax_se_()
    covers = lower <= true_argmax <= upper
    ax.scatter(i, model.argmax_(), color=COLORS[covers], label=lgn)
    ax.plot(
        [i, i], [lower, upper],
        color=COLORS[covers]
    )

ax.legend()

# Coverage

In [ ]:
sim = CoverageSimulation(f).sample(n=10_000, nxs=1000)

In [ ]:
cis = sim.argmax_cis_()

In [ ]:
cis.covers(f.argmax()).mean()

## One-Parameter Reduction

With a, b and α known, the argmax t* = g(β) depends on β alone, and is increasing and convex in this regime. The delta method maps the Wald interval β̂ ± zs through the tangent to g at β̂, while the transformed interval maps it through g itself. A convex function lies above its tangents, so both endpoints of the delta interval fall below those of the transformed interval: a low estimate gets a short interval that misses below.

In [ ]:
from typing import Callable

from matplotlib.axes import Axes
import scipy.stats as sts

from glm.extras.participation_intensity import ArgmaxGradient


def plot_tangent_versus_curve_interval(
    ax: Axes,
    g: Callable[[float], float],
    dg: Callable[[float], float],
    β: float,
    β̂: float,
    s: float,
    level: float = 0.95,
) -> Axes:
    """Compare the delta method interval for g(β) with the transformed interval.

    Both intervals map the Wald interval β̂ ± zs onto the vertical axis: the
    delta method through the tangent line to g at β̂, the transformed interval
    through g itself.

    Parameters
    ----------
    ax : Axes
        The axes to draw on.
    g : Callable[[float], float]
        The function of β whose value is estimated.
    dg : Callable[[float], float]
        The derivative of g.
    β : float
        The true value of the parameter.
    β̂ : float
        The estimate of the parameter.
    s : float
        The standard error of the estimate.
    level : float
        The confidence level of the intervals.

    Returns
    -------
    Axes
        The axes, drawn on.
    """
    z = float(sts.norm.ppf((1 + level) / 2))
    lower = β̂ - z * s
    upper = β̂ + z * s
    curve_ends = (g(lower), g(upper))
    tangent_ends = (g(β̂) + dg(β̂) * (lower - β̂), g(β̂) + dg(β̂) * (upper - β̂))
    t = g(β)

    # Frame the intervals: the β interval and the true β horizontally, both
    # intervals for t* and the true t* vertically.
    β_min = min(lower, β)
    β_max = max(upper, β)
    β_pad = 0.15 * (β_max - β_min)
    t_min = min(*curve_ends, *tangent_ends, t)
    t_max = max(*curve_ends, *tangent_ends, t)
    t_pad = 0.15 * (t_max - t_min)
    ax.set_xlim(β_min - β_pad, β_max + β_pad)
    ax.set_ylim(t_min - t_pad, t_max + t_pad)
    x_left, _ = ax.get_xlim()
    # Clear of the bottom edge, so the Wald interval isn't clipped.
    y_bottom = t_min - 0.6 * t_pad

    βs = np.linspace(β_min - β_pad, β_max + β_pad, num=250)
    ax.plot(βs, [g(b) for b in βs], color="black", linewidth=3, label="g(β)")
    ax.plot(
        βs,
        g(β̂) + dg(β̂) * (βs - β̂),
        color="tab:orange",
        linewidth=3,
        # linestyle="--",
        label="Tangent at β̂",
    )
    ax.axhline(t, color="silver", linestyle="--", linewidth=3, label="True β and t* = g(β)")
    ax.axvline(β, color="silver", linestyle="--", linewidth=3)

    # The Wald interval for β, along the bottom of the axes.
    ax.plot(
        [lower, upper],
        [y_bottom, y_bottom],
        color="tab:blue",
        linewidth=8,
        solid_capstyle="butt",
        label="Wald interval for β",
    )
    ax.scatter([β̂], [y_bottom], color="white", s=20, zorder=3)

    # Each interval for t*, along the left of the axes, traced from the β
    # interval's endpoints through the curve or the tangent.
    β_span = β_max - β_min + 2 * β_pad
    for ends, offset, color, label in (
        (curve_ends, 0.012, "tab:blue", "Transformed interval"),
        (tangent_ends, 0.035, "tab:orange", "Delta method interval"),
    ):
        x = x_left + offset * β_span
        ax.plot(
            [x, x], ends, color=color, linewidth=8, solid_capstyle="butt", label=label
        )
        for β_end, t_end in zip((lower, upper), ends):
            ax.plot([β_end, β_end], [y_bottom, t_end], color=color, linewidth=0.75, linestyle='--', alpha=1.0)
            ax.plot([x, β_end], [t_end, t_end], color=color, linewidth=0.75, linestyle='--', alpha=1.0)

    ax.set_xlabel("β")
    ax.set_ylabel("t")
    return ax

In [ ]:
a, b, α, β = -1.0, 5.0, -0.5, -2.0


def g(x: float) -> float:
    """The argmax as a function of β alone, with a, b, α known."""
    return ParticipationIntensityFunction(a, b, α, x).argmax()


def dg(x: float) -> float:
    return float(ArgmaxGradient(ParticipationIntensityFunction(a, b, α, x)).gradient()[3])


fig, axs = plt.subplots(1, 2, figsize=(14, 5))
plot_tangent_versus_curve_interval(axs[0], g, dg, β, β - 0.55, s=0.33)
axs[0].set_title("Low estimate, β̂ = β − 0.6: delta misses, transformed covers")
plot_tangent_versus_curve_interval(axs[1], g, dg, β, β + 0.55, s=0.33)
axs[1].set_title("High estimate, β̂ = β + 0.5: both cover")
axs[1].legend(loc="upper right")
fig.tight_layout()